This notebook creates the Transfer Learning model

In [ ]:
import torch
import torch.nn as nn
from auxiliary import *
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

seed_everything()

## Data Augmentation Methods


In [2]:
seed_everything()
train_loader, val_loader, test_loader, train_dataset = get_dataloaders(batch_size=64, img_size=308) # Automatically cropped
class_weights = compute_class_weights(train_dataset) # Class weights for clas imbalance 

Class distribution: [  86 1088  535  431  315  163  898]
Class loss weights: [5.8405313  0.46165967 0.9388518  1.1653961  1.5945579  3.0815074
 0.5593382 ]


## DINONet

There are different DINO options

For now, we will start with dinov2_vits14 which has around 21M parameters and its better for small datasets

DINOv2 models are directly accessible through pytorch.

DINOv3 models are installed differently, however, there is one DINOv3 net pretrained specifically on 493M satellite images, which we should check in teh future

In [ ]:
class DINONet(nn.Module):
    def __init__(self, num_classes = NUM_CLASSES, model_name="dinov2_vits14"):
        super().__init__()

        self.backbone = torch.hub.load('facebookresearch/dinov2', model_name)

        embed_dim = self.backbone.embed_dim

    
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Linear probing, prediction head
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

    

In [ ]:
model_ft = DINONet().to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer_ft = optim.AdamW(
    filter(lambda p: p.requires_grad, model_ft.parameters()), 
    lr=FT_LR, 
    weight_decay=1e-4
)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=3, gamma=0.1)


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to C:\Users\19aca/.cache\torch\hub\main.zip


C:\Users\19aca/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\19aca/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\19aca/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to C:\Users\19aca/.cache\torch\hub\checkpoints\dinov2_vits14_pretrain.pth


  0%|          | 128k/84.2M [00:06<1:13:31, 20.0kB/s]


KeyboardInterrupt: 

In [ ]:
trained_dino = train_model(
    model=model_ft,
    criterion=criterion,
    optimizer=optimizer_ft,
    scheduler=exp_lr_scheduler,
    num_epochs=10,
    plot_confusion_matrix=True
)

## Generating Submission

Note this just generates for Transfer Learning, submissions must have both custom and TL.


In [ ]:
def generate_predictions(model, test_loader, output_path):
    model.eval()
    preds = []
    with torch.no_grad():
        for sample in test_loader:
            inputs = sample['image'].to(device)
            logits = model(inputs)
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            
    df = test_loader.dataset.dataset.copy()
    df['prediction'] = preds
    df.to_csv(output_path, index=False)
    print(f"Predictions saved to {output_path}")

generate_predictions(trained_dino, test_loader, OUTPUT_DIR / 'finetuning_predictions.csv')